ChatGPT was used to help generate the code in this notebook

# Introduction

This notebook presents HollywoodIdeaGPT, a decoder-only transformer for conditional text generation in the movie domain. Given high-level movie attributes such as genre, actors, characters, themes, and related metadata, the model aims to generate a coherent plot synopsis.

To train the model, we use a dataset of movie metadata–synopsis pairs. The training objective is based on autoregressive next-token prediction.

# Imports

In [1]:
%pip install torchinfo > /dev/null
%pip install einops > /dev/null
%pip install wandb > /dev/null
%pip install kaggle kagglehub pandas
%pip install tokenizers
%pip install wandb
%pip install scikit-learn

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase, NFD, StripAccents, Sequence

from collections import Counter, defaultdict
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data.dataset import Dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F

import ast
import os
from tqdm.auto import tqdm
import wandb
import math

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


c:\Users\Felix\Downloads\projet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data and tokenization


## Downloading and parsing dataset

Downloading and parsing of the dataset. The dataset used will be "The Movies Dataset" from Kaggle: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv.

In [ ]:
# !kaggle datasets download -d rounakbanik/the-movies-dataset -p data/ --unzip

Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0




  0%|          | 0.00/228M [00:00<?, ?B/s]
  0%|          | 1.00M/228M [00:00<02:09, 1.83MB/s]
  1%|          | 2.00M/228M [00:01<01:52, 2.10MB/s]
  1%|▏         | 3.00M/228M [00:01<01:33, 2.51MB/s]
  2%|▏         | 4.00M/228M [00:01<01:17, 3.04MB/s]
  2%|▏         | 5.00M/228M [00:01<01:12, 3.23MB/s]
  3%|▎         | 6.00M/228M [00:02<01:02, 3.70MB/s]
  3%|▎         | 7.00M/228M [00:02<01:00, 3.82MB/s]
  4%|▎         | 8.00M/228M [00:02<00:56, 4.07MB/s]
  4%|▍         | 9.00M/228M [00:02<00:55, 4.14MB/s]
  4%|▍         | 10.0M/228M [00:03<00:52, 4.32MB/s]
  5%|▍         | 11.0M/228M [00:03<00:51, 4.41MB/s]
  5%|▌         | 12.0M/228M [00:03<00:50, 4.49MB/s]
  6%|▌         | 13.0M/228M [00:03<00:49, 4.58MB/s]
  6%|▌         | 14.0M/228M [00:03<00:48, 4.61MB/s]
  7%|▋         | 15.0M/228M [00:04<00:47, 4.66MB/s]
  7%|▋         | 16.0M/228M [00:04<00:47, 4.69MB/s]
  7%|▋         | 17.0M/228M [00:04<00:46, 4.72MB/s]
  8%|▊         | 18.0M/228M [00:04<00:46, 4.73MB/s]
  8%|▊         | 19.

In [2]:
movies = pd.read_csv("data/movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("data/keywords.csv")
ratings = pd.read_csv("data/ratings.csv")
credits = pd.read_csv("data/credits.csv")

# Clean ids
movies["id"] = pd.to_numeric(movies["id"], errors="coerce")
movies = movies.dropna(subset=["id"]).copy()
movies["id"] = movies["id"].astype("int64")

keywords["id"] = pd.to_numeric(keywords["id"], errors="coerce")
keywords = keywords.dropna(subset=["id"]).copy()
keywords["id"] = keywords["id"].astype("int64")

credits["id"] = pd.to_numeric(credits["id"], errors="coerce")
credits = credits.dropna(subset=["id"]).copy()
credits["id"] = credits["id"].astype("int64")

# Merge tables
movies = movies.merge(keywords, on="id", how="left")
movies = movies.merge(credits, on="id", how="left")

# Drop duplicate id
movies = movies.drop(columns=["movieId"], errors="ignore")

In [3]:

movies = movies[movies['adult'] != "true"].copy()

movies = movies.drop(columns=[
    "adult", "homepage", "imdb_id", "budget", "revenue", "original_language", "release_date",
    "runtime", "vote_average", "vote_count", "popularity", "poster_path", "belongs_to_collection",
    "production_companies", "production_countries", "spoken_languages", "tagline", "video",
    "status", "original_title", "crew"])

movies = movies.dropna().copy()


In [4]:
def parse_json_like(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return []
    return []

for col in ["genres", "keywords", "cast"]:
    movies[col] = movies[col].apply(parse_json_like)

In [5]:
print(f"Dataset size: {movies.size}")
print(f"Dataset columns: {movies.columns}")
print(f"Dataset shape: {movies.shape}")

Dataset size: 273774
Dataset columns: Index(['genres', 'id', 'overview', 'title', 'keywords', 'cast'], dtype='str')
Dataset shape: (45629, 6)


## Dataset class and helper functions

In [6]:
SPECIAL_TOKENS = ["<pad>", "<bos>", "<sep>", "<eos>", "<unk>"]
pad_id = 0

`build_prompt(row, top_cast=5)` :

Builds the metadata prompt for one movie. It extracts the genre names, keyword names, and the top cast members, then formats them into a single text string that will be used as the model input.

---

`yield_training_texts(df)` :

Generates the full training text for the tokenizer from each movie in the dataframe. For every row, it creates a sequence of the form `<bos> prompt <sep> overview <eos>` so the tokenizer learns the same structure the model will see during training.

---

`collate_fn(batch, pad_id)` :

Pads a batch of tokenized sequences to the same length so they can be processed together by the model. It also removes any empty samples before padding.

In [7]:
def build_prompt(row, top_cast=5):
    genres = ", ".join(g['name'] for g in row['genres'])
    keywords = ", ".join(k['name'] for k in row['keywords'])
    cast = ", ".join(actor['name'] for actor in row['cast'][:top_cast])

    return (
        f"title: {row['title']} | "
        f"genres: {genres} | "
        f"keywords: {keywords} | "
        f"cast: {cast}"
    )

def yield_training_texts(df):
    """
    Yield full training strings for BPE tokenizer training.
    We train on the same overall format the decoder will later see.
    """
    for _, row in df.iterrows():
        prompt = build_prompt(row)
        summary = row["overview"]
        yield f"<bos> {prompt} <sep> {summary} <eos>"

def collate_fn(batch):
    input_ids, target_ids, loss_masks = zip(*batch)

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    target_ids = pad_sequence(target_ids, batch_first=True, padding_value=pad_id)
    loss_masks = pad_sequence(loss_masks, batch_first=True, padding_value=0.0)

    attention_mask = (input_ids != pad_id).long()

    return input_ids, target_ids, loss_masks, attention_mask

# Test prompt :
print(build_prompt(movies.iloc[0]))

title: Toy Story | genres: Animation, Comedy, Family | keywords: jealousy, toy, boy, friendship, friends, rivalry, boy next door, new toy, toy comes to life | cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn


`MovieDataset(Dataset)` :

Custom PyTorch dataset for the movie-generation task. For each movie, it builds the metadata prompt and target overview, tokenizes them, combines them into a single decoder-only sequence, truncates to the maximum length, and returns the input tokens, target tokens, and a loss mask so that training focuses only on the summary portion.

In [8]:
class MovieDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = build_prompt(row)
        summary = row["overview"]

        prompt_ids = self.tokenizer.encode(f"<bos> {prompt} <sep>").ids
        summary_ids = self.tokenizer.encode(f"{summary} <eos>").ids

        full_ids = prompt_ids + summary_ids

        if len(full_ids) > self.max_length:
            full_ids = full_ids[:self.max_length]

        input_ids = full_ids[:-1]
        target_ids = full_ids[1:]

        # mask loss only on summary region
        loss_mask = [0] * max(0, len(prompt_ids) - 1) + [1] * max(0, len(full_ids) - len(prompt_ids))

        loss_mask = loss_mask[:len(input_ids)]

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long),
            torch.tensor(loss_mask, dtype=torch.float)
        )

## Training tokenizer / Loading tokenizer

Our dataset contains many rare words, so choosing an appropriate tokenization method is important for achieving good performance. A basic word-level tokenizer would produce many <UNK> tokens and would struggle with movie titles, actor names, character names, and other infrequent terms. On the other hand, character-level tokenization would greatly increase sequence lengths, making training more expensive and reducing the efficiency of the attention mechanism.

For these reasons, we use Byte Pair Encoding (BPE). BPE begins with small text units and progressively merges frequently occurring symbol pairs into larger subword tokens. It reduces the number of unknown tokens while keeping sequence lengths much shorter than character-level tokenization. This provides a good compromise between vocabulary size and flexibility.

We will use a HuggingFace BPE Tokeizer


In [9]:
def train_bpe_tokenizer(
    train_df,
    vocab_size=16000,
    min_frequency=2,
    save_path=None,
):
    """
    Train a BPE tokenizer from the training dataframe.

    Args:
        train_df: pandas DataFrame containing the training rows
        vocab_size: target vocabulary size
        min_frequency: minimum frequency for merges/tokens
        save_path: optional path to save tokenizer JSON

    Returns:
        tokenizer: trained Hugging Face Tokenizer
    """
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.normalizer = Sequence([NFD(), Lowercase(), StripAccents()])
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        special_tokens=SPECIAL_TOKENS,
    )

    tokenizer.train_from_iterator(
        yield_training_texts(train_df),
        trainer=trainer,
    )

    if save_path is not None:
        tokenizer.save(save_path)

    return tokenizer

In [10]:
TOKENIZER_PATH = "movie_bpe_tokenizer.json"

train_df, temp_df = train_test_split(movies, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

if os.path.exists(TOKENIZER_PATH):
    print("Loading existing tokenizer...")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
else:
    print("Training new tokenizer...")
    tokenizer = train_bpe_tokenizer(
        train_df,
        vocab_size=16000,
        min_frequency=2,
        save_path=TOKENIZER_PATH
    )
    tokenizer.save(TOKENIZER_PATH)

print("Tokenizer ready.")

Loading existing tokenizer...
Tokenizer ready.


In [11]:
row = movies.iloc[0]

prompt = build_prompt(row)
summary = row["overview"]

full_text = f"<bos> {prompt} <sep> {summary} <eos>"
encoding = tokenizer.encode(full_text)

print("Full text:")
print(full_text)
print()

print("First 100 tokens:")
print(encoding.tokens[:100])
print()

print("First 100 token IDs:")
print(encoding.ids[:100])
print()

print("Total number of tokens:", len(encoding.ids))

Full text:
<bos> title: Toy Story | genres: Animation, Comedy, Family | keywords: jealousy, toy, boy, friendship, friends, rivalry, boy next door, new toy, toy comes to life | cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn <sep> Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. <eos>

First 100 tokens:
['<bos>', 'title', ':', 'toy', 'story', '|', 'genres', ':', 'animation', ',', 'comedy', ',', 'family', '|', 'keywords', ':', 'jealousy', ',', 'toy', ',', 'boy', ',', 'friendship', ',', 'friends', ',', 'rivalry', ',', 'boy', 'next', 'door', ',', 'new', 'toy', ',', 'toy', 'comes', 'to', 'life', '|', 'cast', ':', 'tom', 'hanks', ',', 'tim', 'allen', ',', 'don', 'rick', 'les', ',', 'jim', 'var', 'ney', ',', 'wallace', 'sha

# Defining the model

In [12]:


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # attention_mask: (B, T) with 1 for real tokens, 0 for pad
        attn_mask = None
        if attention_mask is not None:
            # convert to shape (B, 1, 1, T) for key padding
            attn_mask = attention_mask[:, None, None, :].bool()

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.ln1(x), attention_mask=attention_mask)
        x = x + self.ff(self.ln2(x))
        return x


class MovieGPT(nn.Module):
    def __init__(self, vocab_size, max_length, d_model=256, n_heads=4, n_layers=4, d_ff=1024, dropout=0.1):
        super().__init__()
        self.max_length = max_length
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_length, d_model)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= self.max_length, f"Sequence length {T} exceeds max_length {self.max_length}"

        positions = torch.arange(0, T, device=input_ids.device).unsqueeze(0)

        x = self.token_emb(input_ids) + self.pos_emb(positions)
        x = self.drop(x)

        for block in self.blocks:
            x = block(x, attention_mask=attention_mask)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

# Train Loop

## Train/Valid/Test loaders

In [13]:
pad_id = tokenizer.token_to_id("<pad>")

train_dataset = MovieDataset(train_df, tokenizer, max_length=256)
val_dataset = MovieDataset(val_df, tokenizer, max_length=256)
test_dataset = MovieDataset(test_df, tokenizer, max_length=256)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

## Training loop

In [14]:
def compute_loss(logits, target_ids, loss_masks, attention_mask):
    """
    logits: (B, T, V)
    target_ids: (B, T)
    loss_masks: (B, T) -> 1 on summary tokens, 0 elsewhere
    attention_mask: (B, T) -> 1 on real tokens, 0 on pad
    """
    B, T, V = logits.shape

    loss_per_token = F.cross_entropy(
        logits.view(B * T, V),
        target_ids.view(B * T),
        reduction="none"
    ).view(B, T)

    valid_mask = loss_masks * attention_mask.float()
    loss = (loss_per_token * valid_mask).sum() / valid_mask.sum().clamp_min(1.0)

    return loss

In [15]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    for input_ids, target_ids, loss_masks, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        loss_masks = loss_masks.to(device)
        attention_mask = attention_mask.to(device)

        logits = model(input_ids, attention_mask=attention_mask)
        loss = compute_loss(logits, target_ids, loss_masks, attention_mask)

        total_loss += loss.item()
        total_batches += 1

    avg_loss = total_loss / max(total_batches, 1)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")

    return avg_loss, perplexity

In [16]:

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs=50,
    project_name="HollywoodIdeaGPT",
    run_name=None,
    grad_clip=1.0,
    patience=5,
    min_delta=0.0,
):
    wandb.init(
        project=project_name,
        name=run_name,
        config={
            "num_epochs": num_epochs,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "weight_decay": optimizer.param_groups[0].get("weight_decay", 0.0),
            "grad_clip": grad_clip,
            "patience": patience,
            "min_delta": min_delta,
        }
    )

    wandb.watch(model, log="gradients", log_freq=100)

    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        model.train()

        total_train_loss = 0.0
        total_train_batches = 0

        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for step, (input_ids, target_ids, loss_masks, attention_mask) in enumerate(progress_bar):
            input_ids = input_ids.to(device)
            target_ids = target_ids.to(device)
            loss_masks = loss_masks.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()

            logits = model(input_ids, attention_mask=attention_mask)
            loss = compute_loss(logits, target_ids, loss_masks, attention_mask)

            loss.backward()

            if grad_clip is not None:
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            else:
                grad_norm = 0.0

            optimizer.step()

            total_train_loss += loss.item()
            total_train_batches += 1

            avg_train_loss_so_far = total_train_loss / total_train_batches
            train_ppl_so_far = math.exp(avg_train_loss_so_far) if avg_train_loss_so_far < 20 else float("inf")

            progress_bar.set_postfix(
                train_loss=f"{avg_train_loss_so_far:.4f}",
                train_ppl=f"{train_ppl_so_far:.2f}"
            )

            wandb.log({
                "train_step_loss": loss.item(),
                "train_step_perplexity": math.exp(loss.item()) if loss.item() < 20 else float("inf"),
                "learning_rate": optimizer.param_groups[0]["lr"],
                "grad_norm": float(grad_norm) if not isinstance(grad_norm, float) else grad_norm,
                "epoch": epoch + 1,
            })

        avg_train_loss = total_train_loss / max(total_train_batches, 1)
        train_perplexity = math.exp(avg_train_loss) if avg_train_loss < 20 else float("inf")

        val_loss, val_perplexity = evaluate(model, val_loader, device)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_perplexity": train_perplexity,
            "val_loss": val_loss,
            "val_perplexity": val_perplexity,
            "epochs_without_improvement": epochs_without_improvement,
        })

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"train_loss={avg_train_loss:.4f}, train_ppl={train_perplexity:.2f} | "
            f"val_loss={val_loss:.4f}, val_ppl={val_perplexity:.2f}"
        )

        improved = val_loss < best_val_loss - min_delta

        if improved:
            best_val_loss = val_loss
            epochs_without_improvement = 0

            torch.save(model.state_dict(), "best_movie_gpt.pt")
            wandb.save("best_movie_gpt.pt")

            print(f"New best val_loss: {best_val_loss:.4f}. Model saved.")
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement}/{patience} epochs.")

            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch+1}. Best val_loss: {best_val_loss:.4f}")
                break

    wandb.finish()

In [17]:
# FOR TESTING:
import math
import torch

def overfit_tiny_batch(model, dataloader, optimizer, device, num_epochs=20):
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0.0
        total_batches = 0

        for input_ids, target_ids, loss_masks, attention_mask in dataloader:
            input_ids = input_ids.to(device)
            target_ids = target_ids.to(device)
            loss_masks = loss_masks.to(device)
            attention_mask = attention_mask.to(device)

            optimizer.zero_grad()

            logits = model(input_ids, attention_mask=attention_mask)
            loss = compute_loss(logits, target_ids, loss_masks, attention_mask)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()
            total_batches += 1

        avg_loss = total_loss / max(total_batches, 1)
        ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")

        print(f"Epoch {epoch+1:02d} | loss={avg_loss:.4f} | ppl={ppl:.2f}")

# Model & tests

## Test to see max size

In [18]:
lengths = []

for _, row in train_df.iterrows():
    prompt = build_prompt(row)
    summary = row["overview"]
    full_text = f"<bos> {prompt} <sep> {summary} <eos>"
    lengths.append(len(tokenizer.encode(full_text).ids))

import numpy as np
print("mean:", np.mean(lengths))
print("median:", np.median(lengths))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))
print("max:", np.max(lengths))

mean: 118.24765087801002
median: 110.0
90th percentile: 186.0
95th percentile: 212.0
99th percentile: 257.0
max: 633


##  Basic overfit test

In [22]:
# OVERFIT TEST
from torch.utils.data import Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
tiny_size = 64
tiny_dataset = Subset(train_dataset, list(range(tiny_size)))

tiny_loader = DataLoader(
    tiny_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

tiny_model = MovieGPT(
    vocab_size=tokenizer.get_vocab_size(),
    max_length=256,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    dropout=0.1,
).to(device)

tiny_optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=3e-4, weight_decay=0.01)

overfit_tiny_batch(
    model=tiny_model,
    dataloader=tiny_loader,
    optimizer=tiny_optimizer,
    device=device,
    num_epochs=20,
)

cuda
Epoch 01 | loss=9.7884 | ppl=17826.56
Epoch 02 | loss=9.5601 | ppl=14187.14
Epoch 03 | loss=9.2293 | ppl=10191.20
Epoch 04 | loss=8.7223 | ppl=6138.21
Epoch 05 | loss=8.1792 | ppl=3565.95
Epoch 06 | loss=7.7220 | ppl=2257.46
Epoch 07 | loss=7.3686 | ppl=1585.34
Epoch 08 | loss=7.0629 | ppl=1167.87
Epoch 09 | loss=6.8155 | ppl=911.86
Epoch 10 | loss=6.6509 | ppl=773.47
Epoch 11 | loss=6.5008 | ppl=665.70
Epoch 12 | loss=6.4097 | ppl=607.69
Epoch 13 | loss=6.3478 | ppl=571.25
Epoch 14 | loss=6.2997 | ppl=544.41
Epoch 15 | loss=6.2432 | ppl=514.50
Epoch 16 | loss=6.2105 | ppl=497.96
Epoch 17 | loss=6.1768 | ppl=481.43
Epoch 18 | loss=6.1223 | ppl=455.90
Epoch 19 | loss=6.0814 | ppl=437.63
Epoch 20 | loss=6.0375 | ppl=418.86


## The real model

In [25]:
vocab_size = 16_000
max_length = 256

model = MovieGPT(
    vocab_size=vocab_size,
    max_length=max_length,
    d_model=384,
    n_heads=6,
    n_layers=6,
    d_ff=1536,
    dropout=0.1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

Device: cuda
Number of parameters: 23033856


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_NAME = "gpt-small-6layers-6heads-384dmodel-1536dff"
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

run = wandb.init(project="HollywoodIdeaGPT", name=RUN_NAME)


train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    num_epochs=50,
    project_name="HollywoodIdeaGPT",
    run_name=RUN_NAME,
    grad_clip=1.0,
    patience=5,
    min_delta=0.0002,
)

# Save final model too, not just best model
torch.save(model.state_dict(), "final_movie_gpt.pt")
print("Saved final model to final_movie_gpt.pt")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Felix\_netrc.
wandb: Currently logged in as: scoobyfelix (scoobyfelix-polytechnique-montreal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/50: 100%|██████████| 1141/1141 [01:57<00:00,  9.70it/s, train_loss=6.4132, train_ppl=609.87]
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


Epoch 1/50 | train_loss=6.4132, train_ppl=609.87 | val_loss=5.9723, val_ppl=392.40
New best val_loss: 5.9723. Model saved.


Epoch 2/50: 100%|██████████| 1141/1141 [01:55<00:00,  9.84it/s, train_loss=5.8016, train_ppl=330.84]


Epoch 2/50 | train_loss=5.8016, train_ppl=330.84 | val_loss=5.6430, val_ppl=282.32
New best val_loss: 5.6430. Model saved.


...

Epoch 16/50 | train_loss=4.1640, train_ppl=64.33 | val_loss=5.0544, val_ppl=156.70
No improvement for 4/5 epochs.


Epoch 17/50: 100%|██████████| 1141/1141 [01:53<00:00, 10.09it/s, train_loss=4.1216, train_ppl=61.66]


Epoch 17/50 | train_loss=4.1216, train_ppl=61.66 | val_loss=5.0606, val_ppl=157.69
No improvement for 5/5 epochs.
Early stopping at epoch 17. Best val_loss: 5.0299


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇▇█████
epochs_without_improvement,▁▁▁▁▁▁▁▁▁▁▁▁▁▃▅▆█
grad_norm,▁▁▂▂▂▃▃▄▄▄▅▄▅▆▅▅▆▅▆▆▅▆▆▆▇▆▆▆▆▇▅▆▆▆▆▆▇▆█▆
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁
train_perplexity,█▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁
train_step_loss,█▇▇▆▆▄▃▃▃▃▃▂▃▂▃▂▂▃▂▃▂▂▂▂▁▂▂▂▂▁▁▂▂▁▁▂▁▁▁▁
train_step_perplexity,█▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▆▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁
val_perplexity,█▅▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁
epoch,17


Saved final model to final_movie_gpt.pt


In [21]:
model.load_state_dict(torch.load("best_movie_gpt.pt", map_location=device))
model.eval()

C:\Users\Felix\AppData\Local\Temp\ipykernel_84060\1057214460.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_movie_gpt.pt", map_lo

MovieGPT(
  (token_emb): Embedding(16000, 256)
  (pos_emb): Embedding(256, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x GPTBlock(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=256, out_features=256, bias=True)
        (k_proj): Linear(in_features=256, out_features=256, bias=True)
        (v_proj): Linear(in_features=256, out_features=256, bias=True)
        (out_proj): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (ln_f): LayerN

In [22]:
@torch.no_grad()
def generate_summary(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=80,
    max_length=512,
):
    model.eval()

    bos_id = tokenizer.token_to_id("<bos>")
    sep_id = tokenizer.token_to_id("<sep>")
    eos_id = tokenizer.token_to_id("<eos>")

    # Start from prompt only
    input_ids = tokenizer.encode(f"<bos> {prompt} <sep>").ids
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Keep only the last max_length tokens if needed
        if input_ids.size(1) > max_length:
            input_ids = input_ids[:, -max_length:]

        attention_mask = torch.ones_like(input_ids, device=device)

        logits = model(input_ids, attention_mask=attention_mask)
        next_token_logits = logits[:, -1, :]              # last position
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)  # greedy

        input_ids = torch.cat([input_ids, next_token_id], dim=1)

        if next_token_id.item() == eos_id:
            break

    generated_ids = input_ids[0].tolist()

    # Remove the prompt part from the decoded output
    prompt_len = len(tokenizer.encode(f"<bos> {prompt} <sep>").ids)
    summary_ids = generated_ids[prompt_len:]

    # Cut at eos if present
    if eos_id in summary_ids:
        summary_ids = summary_ids[:summary_ids.index(eos_id)]

    return tokenizer.decode(summary_ids)

In [23]:
test_prompt = (
    "title: Shadow Protocol | "
    "genres: Action, Thriller, Science Fiction | "
    "keywords: espionage, AI, betrayal, conspiracy | "
    "cast: Rebecca Ferguson, Oscar Isaac, John David Washington"
)

generated = generate_summary(
    model=model,
    tokenizer=tokenizer,
    prompt=test_prompt,
    device=device,
    max_new_tokens=80,
    max_length=512,
)

print("PROMPT:")
print(test_prompt)
print("\nGENERATED SUMMARY:")
print(generated)

PROMPT:
title: Shadow Protocol | genres: Action, Thriller, Science Fiction | keywords: espionage, AI, betrayal, conspiracy | cast: Rebecca Ferguson, Oscar Isaac, John David Washington

GENERATED SUMMARY:
a man is forced to take a mission to save his life .


In [24]:
for i in range(5):
    row = test_df.iloc[i]
    prompt = build_prompt(row)

    generated = generate_summary(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=80,
        max_length=512,
    )

    print("=" * 80)
    print(f"Example {i+1}")
    print("\nPROMPT:")
    print(prompt)
    print("\nTRUE OVERVIEW:")
    print(row["overview"])
    print("\nGENERATED OVERVIEW:")
    print(generated)
    print()

Example 1

PROMPT:
title: Life 2.0 | genres: Documentary | keywords: video game, avatars, crafting, game commerce, virtual relationship | cast: 

TRUE OVERVIEW:
This feature-length documentary follows a group of people whose lives are dramatically transformed by a virtual world -- reshaping relationships, identities, and ultimately the very notion of reality.

GENERATED OVERVIEW:
a documentary about the life of a man who was killed in a small town in the country .

Example 2

PROMPT:
title: A Sun-Tribe Myth from the Bakumatsu Era | genres: Comedy | keywords:  | cast: Frankie Sakai, Sachiko Hidari, Yôko Minamida, Yûjirô Ishihara, Izumi Ashikawa

TRUE OVERVIEW:
Saheji, a man-about-town, gets stuck at a high-class brothel when he can’t pay the bill. He makes the best of his situation by performing various tasks amidst the tumult of the end of the shogunate—but always by making sure to get a “commission” for his troubles.

GENERATED OVERVIEW:
a young man is forced to take his family to liv